# Lab 03.4 – XADC Thermal Response Under CPU Load

## Objectives
- Observe the Zynq die temperature before, during and after processor load.
- Generate a controlled CPU workload from Python.
- Record the thermal response as a function of time.
- Relate computational activity to power dissipation and temperature.

## Hardware
- Digilent Zybo legacy board
- PYNQ 3.0.1
- No external sensor or additional hardware is required.

## Background
The XADC temperature sensor measures the silicon die temperature, not the room temperature. When processor activity increases, dynamic power dissipation generally increases. Because the package and PCB have thermal mass, the temperature response is gradual rather than instantaneous.

This experiment uses two short-lived Python worker processes to load the ARM cores. The workload is intentionally finite and is stopped automatically in a `finally` block.


## Exercise 1 – Initialize temperature measurement


In [ ]:
from pathlib import Path
import time
import multiprocessing as mp
import statistics
import matplotlib.pyplot as plt

iio_root = Path("/sys/bus/iio/devices")
xadc_path = None

for dev in sorted(iio_root.glob("iio:device*")):
    name_file = dev / "name"
    if name_file.exists() and name_file.read_text().strip() == "xadc":
        xadc_path = dev
        break

if xadc_path is None:
    raise RuntimeError("XADC IIO device not found.")

def read_temperature_celsius():
    raw = float((xadc_path / "in_temp0_raw").read_text().strip())
    offset = float((xadc_path / "in_temp0_offset").read_text().strip())
    scale = float((xadc_path / "in_temp0_scale").read_text().strip())
    return (raw + offset) * scale / 1000.0

print(f"Current die temperature: {read_temperature_celsius():.2f} °C")


## Exercise 2 – Define a finite CPU workload

Each worker repeatedly performs floating-point arithmetic until the shared stop event is set. On the Zybo Zynq-7000 device, two workers normally occupy the two ARM Cortex-A9 cores.


In [ ]:
def cpu_worker(stop_event):
    x = 1.0
    while not stop_event.is_set():
        for _ in range(10000):
            x = (x * 1.000001 + 0.000001) % 1000.0

def start_cpu_load(workers=2):
    stop_event = mp.Event()
    processes = [mp.Process(target=cpu_worker, args=(stop_event,)) for _ in range(workers)]
    for p in processes:
        p.start()
    return stop_event, processes

def stop_cpu_load(stop_event, processes):
    stop_event.set()
    for p in processes:
        p.join(timeout=2)
        if p.is_alive():
            p.terminate()
            p.join()

print("CPU-load functions defined.")


## Exercise 3 – Measure baseline, load and cooldown

The default experiment uses:

- 10 s baseline,
- 30 s CPU load,
- 20 s cooldown,
- one sample per second.

The whole experiment takes approximately one minute. If the cell is interrupted, the worker processes are stopped by the `finally` block.


In [ ]:
BASELINE_TIME = 10
LOAD_TIME = 30
COOLDOWN_TIME = 20
INTERVAL = 1.0
WORKERS = 2

times = []
temperatures = []
phases = []

experiment_start = time.monotonic()

def sample_phase(name, duration):
    samples = int(duration / INTERVAL)
    phase_start = time.monotonic()
    for i in range(samples):
        target = phase_start + i * INTERVAL
        delay = target - time.monotonic()
        if delay > 0:
            time.sleep(delay)

        t = time.monotonic() - experiment_start
        temp = read_temperature_celsius()
        times.append(t)
        temperatures.append(temp)
        phases.append(name)
        print(f"{name:<8} t={t:6.1f} s   T={temp:6.2f} °C")

stop_event = None
processes = []

try:
    sample_phase("baseline", BASELINE_TIME)

    stop_event, processes = start_cpu_load(WORKERS)
    sample_phase("load", LOAD_TIME)

finally:
    if stop_event is not None:
        stop_cpu_load(stop_event, processes)

sample_phase("cooldown", COOLDOWN_TIME)

print("Experiment complete.")


## Exercise 4 – Compare the three phases


In [ ]:
for phase in ("baseline", "load", "cooldown"):
    values = [t for t, p in zip(temperatures, phases) if p == phase]
    if values:
        print(f"{phase:<8}: mean={statistics.mean(values):.2f} °C, min={min(values):.2f} °C, max={max(values):.2f} °C")

if temperatures:
    print(f"\nOverall temperature change: {temperatures[-1] - temperatures[0]:+.2f} °C")
    print(f"Peak above first sample:     {max(temperatures) - temperatures[0]:+.2f} °C")


## Exercise 5 – Plot the thermal response


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(times, temperatures, marker="o")
plt.axvline(BASELINE_TIME, linestyle="--", label="CPU load starts")
plt.axvline(BASELINE_TIME + LOAD_TIME, linestyle="--", label="CPU load stops")
plt.xlabel("Time (s)")
plt.ylabel("Temperature (°C)")
plt.title("Zynq Die Temperature Under CPU Load")
plt.grid(True)
plt.legend()
plt.show()


## Questions
1. Did the die temperature rise immediately when the CPU load started? Explain the observed delay.
2. Why can the temperature continue to rise for some time after the workload changes?
3. Why does the cooldown curve normally differ from an instantaneous step back to the original temperature?
4. Why might different Zybo boards show different absolute temperatures under the same software workload?
5. Does this experiment measure FPGA fabric power directly? Explain.
6. Why is a `finally` block important when generating background CPU load from a Jupyter notebook?
